# LCA Lasers4MaaS - Pre and Post processes



In [48]:
import bw2data as bd
import bw2io as bi
from bw2io import BW2Package
from bw2calc import LCA
from pathlib import Path
import bw2analyzer as ba
import pandas as pd
from IPython.display import display


### Constants: PROJECT_NAME


In [49]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"

bd.projects.set_current(PROJECT_NAME)
if ECOINVENT_DATABASE_NAME not in bd.databases:
    bi.import_ecoinvent_release(version="3.12", system_model="cutoff")
else:
    print(f"Database {ECOINVENT_DATABASE_NAME} already exists, skipping import.")

Database ecoinvent-3.12-cutoff already exists, skipping import.


### Variables: ei


In [50]:
ei = bd.Database(ECOINVENT_DATABASE_NAME)

### Variables: bio


In [51]:
bio = bd.Database("ecoinvent-3.12-biosphere")

### Function: find_ecoinvent_activity


In [52]:
def find_ecoinvent_activity(label, search_terms, exclusion_terms=None, locations=None):
    """Return one ecoinvent activity or issue a warning if selection is ambiguous."""
    
    matches = [
        act for act in ei
        if label.lower() in act.get('name').lower()
    ]

    if not matches:
        matches = list(ei.search(" ".join(search_terms)))
        matches = [activity for activity in matches if all(term in activity.get("name") for term in search_terms)]
        if exclusion_terms is not None:
            matches = [activity for activity in matches if all(term not in activity.get("name") for term in exclusion_terms)]

    if locations is not None:
        matches = [activity for activity in matches if all(region in activity.get("location") for region in locations)]
        
    if not matches:
        matches = list(ei.search(" ".join(search_terms)))
        matches = [activity for activity in matches if any(term in activity.get("name") for term in search_terms)]
            
    if locations is not None:
        for location in locations:
            location_matches = [
                activity for activity in matches
                if activity.get("location") == location
            ]
            
            if len(location_matches) == 1:
                return location_matches[0]
            if len(location_matches) > 1:
                # warnings.warn(
                #     f"Multiple activities match {label!r} in {location!r}: "
                #     f"{location_matches}"
                # )
                return location_matches

    # If exactly one match remains, return it
    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        return matches

    return None

### Call: find_ecoinvent_activity


In [53]:
find_ecoinvent_activity(
    "abrasive blasting, alumina, carbon steel substrate", ("aluminium oxide", "production")
)


['market for abrasive blasting, alumina, carbon steel substrate' (square meter, GLO, None),
 'abrasive blasting, alumina, carbon steel substrate' (square meter, GLO, None)]

### Global parameters

In [54]:
# Technosphere exchanges
# ----------------

# ENERGY
electricity = find_ecoinvent_activity(
    "market for electricity", ("market for electricity", "medium voltage"), locations=["RER"]
)

# WATER
water = find_ecoinvent_activity(
    "tap water", ("market for tap water"), ("treatment"), locations=["RER"]
)

# CHEMICALS
chemical_alkaline = find_ecoinvent_activity(
    "sodium hydroxide, without water, in 50% solution state", ("sodium hydroxide 50% solution state"), locations=["RER"]
)
acid_pickling = find_ecoinvent_activity(
    "hydrochloric acid, without water, in 30% solution state", ("hydrochloric acid", "30% solution state", "production"), locations=["RER"]
)
nitric_acid = find_ecoinvent_activity(
    "nitric acid, without water, in 50% solution state", ("nitric acid", "50% solution state", "production"), locations=["RER"]
)

hydrochloric_acid = find_ecoinvent_activity(
    "hydrochloric acid, without water, in 30% solution state", ("hydrochloric acid", "30% solution state", "production"), locations=["RER"])

### ABRASIVES & CONSUMABLES


In [55]:
# ABRASIVES & CONSUMABLES
abrasive = find_ecoinvent_activity(
    "market for abrasive blasting, alumina, carbon steel substrate", ("aluminium oxide", "production")
)
grinding_wheel = find_ecoinvent_activity(
    "grinding wheel", ("grinding wheel", "conventional", "production"), locations=["RER"]
)
polishing_compound = find_ecoinvent_activity(
    "aluminium oxide polishing", ("aluminium oxide", "production"), locations=["RER"]
)

# COOLANTS
coolant = find_ecoinvent_activity(
    "metal working fluid", ("metal working fluid", "production"), locations=["RER"]
)

### WASTE MANAGEMENT


In [56]:

# WASTE MANAGEMENT
waste_treatment = find_ecoinvent_activity(
    "waste steel landfill", ("treatment", "waste steel", "inert material landfill"), locations=["RER"]
)
waste_paint_treatment = find_ecoinvent_activity(
    "waste paint incineration", ("treatment", "waste paint", "hazardous waste incineration", "with energy recovery"), locations=["RER"]
)
waste_spent_solvent_treatment = find_ecoinvent_activity(
    "spent solvent incineration", ("treatment", "spent solvent mixture", "hazardous waste incineration"), locations=["RER"]
)
waste_spent_solvent_energy_recovery = find_ecoinvent_activity(
    "spent solvent energy recovery", ("treatment", "spent solvent mixture", "hazardous waste incineration", "with energy recovery"), locations=["RER"]
)

### Cost factors for the main resource inputs (adjust as needed)


In [57]:
# Cost factors for the main resource inputs (adjust as needed)
COST_FACTORS = {
    'electricity_EUR_per_kWh': 0.29,     # EUR 0.29/kWh for industrial electricity
    'argon_EUR_per_kg': 5.87,           # EUR 2.5/kg for argon gas
    'nitrogen_EUR_per_kg': 5.35           # EUR 1.2/kg for nitrogen gas
}

### Default industrial tank sizes (liters) for chemical processes


In [58]:
# Default industrial tank sizes (liters) for chemical processes
INDUSTRIAL_TANK_SIZES = {
    'pickling': 10000,           # 10 m³ - standard for joining prep (ASTM A380)
    'alkaline_degreasing': 7500, # 7.5 m³ - common for degreasing
    'passivation': 7500,          # 7.5 m³ - similar to degreasing
    'chemical_etching': 7500,    # 7.5 m³ - similar to pickling
    'conversion_coating': 3000,  # 3 m³ - smaller for dip coating
}

### Get only the most used EF 3.1 methods


In [59]:
# Get only the most used EF 3.1 methods
most_used_indicators = [
    'climate change: total',
    'particulate matter formation',
    'acidification',
    'eutrophication: terrestrial',
    'photochemical oxidant formation: human health',
    'human toxicity: carcinogenic'
]

ef31_methods = [
    m for m in bd.methods
    if ('EF v3.1' in str(m) or 'EF 3.1' in str(m))
    and 'no LT' not in str(m)
    and any(indicator in str(m) for indicator in most_used_indicators)
]

print(f"Found {len(ef31_methods)} most-used EF 3.1 methods")
print("Selected EF 3.1 impact categories:")
for m in ef31_methods:
    print(f"  {m}")

Found 8 most-used EF 3.1 methods
Selected EF 3.1 impact categories:
  ('ecoinvent-3.12', 'EF v3.1', 'acidification', 'accumulated exceedance (AE)')
  ('ecoinvent-3.12', 'EF v3.1', 'eutrophication: terrestrial', 'accumulated exceedance (AE)')
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic', 'comparative toxic unit for human (CTUh)')
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, inorganics', 'comparative toxic unit for human (CTUh)')
  ('ecoinvent-3.12', 'EF v3.1', 'human toxicity: carcinogenic, organics', 'comparative toxic unit for human (CTUh)')
  ('ecoinvent-3.12', 'EF v3.1', 'particulate matter formation', 'impact on human health')
  ('ecoinvent-3.12', 'EF v3.1', 'photochemical oxidant formation: human health', 'tropospheric ozone concentration increase')
  ('ecoinvent-3.12', 'EN15804+A2 - Core impact categories and indicators', 'climate change: total (EF v3.1 - IPCC 2021)', 'global warming potential (GWP100)')


### Calculation engine for pre and post processes

In [60]:
def calculate_process_cost(process, scenario="amount", cost_factors=None):
    """Calculate consumable cost (EUR) from exchange amounts.

    Only electricity (kWh), argon and nitrogen (kg) have a price in cost_factors;
    chemicals, water and abrasives are not costed yet.
    """
    if cost_factors is None:
        cost_factors = COST_FACTORS

    total_cost = 0.0

    for exchange in process.exchanges():
        if exchange["type"] == "technosphere":
            # Get amount for chosen scenario
            if scenario == "minimum":
                amount = exchange.get("minimum", exchange.get("min", exchange["amount"]))
            elif scenario == "maximum":
                amount = exchange.get("maximum", exchange.get("max", exchange["amount"]))
            else:
                amount = exchange["amount"]

            # Identify resource type and calculate cost
            input_node = exchange.input
            input_str = str(input_node).lower()

            if "electricity" in input_str:
                total_cost += amount * cost_factors.get('electricity_EUR_per_kWh', 0.29)
            elif "argon" in input_str:
                total_cost += amount * cost_factors.get('argon_EUR_per_kg', 5.87)
            elif "nitrogen" in input_str:
                total_cost += amount * cost_factors.get('nitrogen_EUR_per_kg', 5.35)
    return total_cost

### Common Uvalues (W/m²·K)


In [61]:
# ======================
# Common U-values (W/m²·K)
# ======================
U_BARE_STEEL = 9.8      # Uninsulated steel tank
U_STAINLESS_STEEL = 8.5
U_FIBERGLASS_5CM = 0.57 # 5 cm fiberglass insulation
U_POLYURETHANE_5CM = 0.45
U_TYPICAL_INSULATED = 0.6

U_VALUES = {
    "bare_steel": U_BARE_STEEL,
    "stainless_steel": U_STAINLESS_STEEL,
    "fiberglass_5cm": U_FIBERGLASS_5CM,
    "polyurethane_5cm": U_POLYURETHANE_5CM,
    "typical_insulated": U_TYPICAL_INSULATED
}

def tank_heat_loss(volume, fluid_temp, room_temp, u_value, time_hours=1):
    """
    Calculate heat loss for a cubic fluid tank.

    Args:
        volume (m³): Tank volume
        fluid_temp (°C): fluid temperature
        room_temp (°C): Room temperature
        u_value (W/m²·K): Heat transfer coefficient
        time_hours (h): Time duration (default: 1)

    Returns:
        (heat_loss_kWh, temp_drop_°C, final_temp_°C)
    """
    side = volume ** (1/3)
    area = 6 * side**2
    mass = volume * 1000
    delta_t = fluid_temp - room_temp
    heat_loss_kwh = u_value * area * delta_t * time_hours / 1000
    temp_drop = heat_loss_kwh / (mass * 4186)
    heat_maintenance_kwh = heat_loss_kwh / 0.95 # 95% heat element efficiency
    return heat_loss_kwh, temp_drop, fluid_temp - temp_drop, heat_maintenance_kwh

## Chemical Pre and Post processes

In [62]:
CHEM_PREPOST_NAME = "Chemical_Pre_and_Post_processes"
CHEM_PREPOST = bd.Database(CHEM_PREPOST_NAME)

# Pre-/post-joining process parameters with min/max ranges.
# Each process appears once; material-specific fields are keyed by "steel" / "aluminium".
# Fields identical for both materials (e.g. description) are stored as plain values.
CHEM_PREPOST_PARAMS = {
"alkaline_degreasing": {
            "standard": {
                "steel": "ASTM B183",
                "aluminium": "ASTM B254",
            },
            "parameters": {
                "steel": {
                    "acid_concentration_%": {"min": 0.05, "max": 0.1},
                    "water_consumption_l": {"min": 5.0, "max": 15.0},
                    "temperature_C": {"min": 60, "max": 80},
                    "time_min": {"min": 5, "max": 15},
                },
                "aluminium": {
                    "acid_concentration_%": {"min": 0.03, "max": 0.1},
                    "water_consumption_l": {"min": 3.0, "max": 10.0},
                    "temperature_C": {"min": 50, "max": 70},
                    "time_min": {"min": 8, "max": 20},
                },
            },
            "description": {
                "steel": "Remove oils, grease, and contaminants from steel surface",
                "aluminium": "Remove oxides and contaminants using alkaline solutions with inhibitors",
            },
        },
"pickling": {
            "standard": "ASTM A380, ISO 16048",
            "parameters": {
                "steel": {
                    "acid_concentration_%": {"min": .1, "max": 0.2},
                    "hydrofloric_acid_consumption_%": {"min": 0.01, "max": 0.02},
                    "water_consumption_l": {"min": 10.0, "max": 25.0},
                    "time_min": {"min": 10, "max": 30},
                    "temperature_C": {"min": 20, "max": 50},
                },
            },
            "description": "Remove weld scale and oxides using acid solutions",
        },
"passivation": {
            "standard": "ASTM A380, ASTM A967",
            "parameters": {
                "steel": {
                    "acid_concentration_%": {"min": 0.2, "max": 0.4},
                    "water_consumption_l": {"min": 5.0, "max": 10.0},
                    "time_min": {"min": 20, "max": 30},
                    "temperature_C": {"min": 20, "max": 50},
                },
            },
            "description": "Form protective oxide layer on stainless steel surface",
        },
}

### Function: create_chemical_process_exchanges


In [63]:
BATH_DENSITY_KG_PER_L = 1.0  # assumed density of the dilute process baths (water-like)


def create_chemical_process_exchanges(
    database_name,
    process_params,
    process_name,
    solution_volume_l=None,
    service_name=None,
):
    """Build Brightway exchanges from CHEM_PREPOST_PARAMS.

    `process_name` is the unique process key (service + material); `service_name`
    (default: `process_name`) selects the chemical and the default tank size.
    `solution_volume_l` is a number in litres (default: INDUSTRIAL_TANK_SIZES).
    `acid_concentration_%` values are mass fractions of active chemical in the
    bath: 0.05 means 5 wt%. Chemical inputs are kg of active substance, matching
    the ecoinvent "without water" datasets. Bath density is BATH_DENSITY_KG_PER_L.
    """
    service_name = service_name or process_name

    default_flows = {
        "water": globals().get("water"),
        "chemical_alkaline": globals().get("chemical_alkaline"),
        "nitric_acid": globals().get("nitric_acid"),
        "electricity": globals().get("electricity"),
        "acid_pickling": globals().get("acid_pickling"),
        "hydrochloric_acid": globals().get("hydrochloric_acid")
    }

    def as_key(activity_or_key):
        return getattr(activity_or_key, "key", activity_or_key)

    def bounds(value):
        if isinstance(value, dict):
            return float(value["min"]), float(value["max"])
        value = float(value)
        return value, value

    exchanges = [
            {
                "input": (database_name, process_name),
                "amount": 1,
                "type": "production",
            }
        ]

    acid = {
        "pickling": "acid_pickling",
        "alkaline_degreasing": "chemical_alkaline",
        "passivation": "nitric_acid",
    }[service_name]

    if solution_volume_l is None:
        solution_volume_l = INDUSTRIAL_TANK_SIZES.get(service_name, 7500)  # default 7.5 m3

    time_min, time_max = bounds(process_params["time_min"])
        
    volume_min, volume_max = bounds(solution_volume_l)
    
    replenishment_factor = 0.001 # fraction of the bath's active chemical replenished per hour (losses, degradation)

    # acid mass (kg of active substance): wt fraction * bath mass * replenishment per hour * hours
    concentration_min, concentration_max = bounds(process_params["acid_concentration_%"])
    acid_min = concentration_min * volume_min * BATH_DENSITY_KG_PER_L * replenishment_factor * time_min / 60
    acid_max = concentration_max * volume_max * BATH_DENSITY_KG_PER_L * replenishment_factor * time_max / 60
    exchanges.append({
        "input": as_key(default_flows[acid]),
        "amount": (acid_max + acid_min)/2,
        "type": "technosphere",
        "minimum": acid_min,
        "maximum": acid_max,
        "uncertainty_type": 1,
    })


    # electricity for heat maintenance
    temp_min, temp_max = bounds(process_params["temperature_C"])
    electricity_kwh_min = tank_heat_loss(solution_volume_l/1000, temp_min, room_temp=20, u_value=U_VALUES["typical_insulated"], time_hours=time_min/60)
    electricity_kwh_max = tank_heat_loss(solution_volume_l/1000, temp_max, room_temp=20, u_value=U_VALUES["typical_insulated"], time_hours=time_max/60)
    exchanges.append({
        "input": as_key(default_flows["electricity"]),
        "amount":(electricity_kwh_max[3] + electricity_kwh_min[3])/2,
        "type": "technosphere",
        "minimum": electricity_kwh_min[3],
        "maximum": electricity_kwh_max[3],
        "uncertainty_type": 1,
    })

    # ionized water consumption rinsing
    water_consumption_min, water_consumption_max = bounds(process_params["water_consumption_l"]) 
    
    exchanges.append({
        "input": as_key(default_flows["water"]),
        "amount": (water_consumption_max + water_consumption_min)/2,
        "type": "technosphere",
        "minimum": water_consumption_min,
        "maximum": water_consumption_max,
        "uncertainty_type": 1,
    })
    return exchanges


ChemProcess_data = {}

for service_name in CHEM_PREPOST_PARAMS:
    record = CHEM_PREPOST_PARAMS.get(service_name)
    materials = list(record.get("parameters", {}).keys())
    for material in materials:
        process_by_material_data = record.get("parameters", {}).get(material)
        process_key = f"{service_name}_{material}"  # one process per service AND material
        ChemProcess_data[(CHEM_PREPOST_NAME, process_key)] = {
            "name": f"{service_name} for {material}",
            "unit": "unit",
            "location": "NL",
            "exchanges": create_chemical_process_exchanges(
                CHEM_PREPOST_NAME,
                process_by_material_data,
                process_key,
                service_name=service_name,
            ),
        }
    

### Call: CHEM_PREPOST.write


In [64]:
CHEM_PREPOST.write(ChemProcess_data)

100%|██████████| 4/4 [00:00<00:00, 85163.53it/s]

10:47:52+0100 [info     ] Vacuuming database            


## Mechanical pre and post processes

In [65]:
MECH_PREPOST_NAME = "Mechanical_Pre_and_Post_processes"
MECH_PREPOST = bd.Database(MECH_PREPOST_NAME)

# Mechanical pre-/post-joining treatments.
MECH_PREPOST_PARAMS = {
"mechanical_treatment": {
            "standard": {
                "steel": "ISO 8501-1 (surface preparation)",
                "aluminium": "ISO 8501-1",
            },
            "parameters": {
                "steel": {
                    "abrasive_consumption_kg": {"min": 0.01, "max": 0.05},
                    "electricity_kwh": {"min": 0.1, "max": 0.4},
                    "time_min": {"min": 5, "max": 20},
                    "surface_area_m2": {"min": 0.5, "max": 2.0},
                },
                "aluminium": {
                    "abrasive_consumption_kg": {"min": 0.005, "max": 0.02},
                    "coolant_consumption_l": {"min": 0.5, "max": 2.0},
                    "electricity_kwh": {"min": 0.05, "max": 0.2},
                    "time_min": {"min": 3, "max": 15},
                    "surface_area_m2": {"min": 0.3, "max": 1.5},
                },
            },
            "condition": {
                "steel": "if oxides/scale are present",
            },
            "note": {
                "aluminium": "Use non-ferrous tools with coolant",
            },
            "description": {
                "steel": "Remove oxides and scale using abrasive methods",
                "aluminium": "Surface preparation using non-ferrous tools with mandatory coolant",
            },
        },
"machining": {
            "standard": "ISO 3685 (tool life)",
            "parameters": {
                "steel": {
                    "machining_time_min": {"min": 1, "max": 5},
                    "tool_wear_pct": {"min": 5, "max": 15},
                    "electricity_kwh": {"min": 0.05, "max": 0.2},
                },
            },
            "condition": "if tight tolerances are required",
            "description": "Precision machining for tight tolerance requirements",
        },
"grinding_polishing":{
            "standard": {
                "steel": "ISO 16048, ASTM B912 (electropolishing)",
                "aluminium": "ISO 16048, ASTM B209",
            },
            "parameters": {
                "steel": {
                    "grinding_wheel_consumption_kg": {"min": 0.005, "max": 0.02},
                    "polishing_compound_kg": {"min": 0.01, "max": 0.05},
                    "electricity_kwh": {"min": 0.1, "max": 0.5},
                    "time_min": {"min": 5, "max": 25},
                    "surface_roughness_um": {"min": 0.1, "max": 1.6},
                },
                "aluminium": {
                    "grinding_wheel_consumption_kg": {"min": 0.002, "max": 0.01},
                    "polishing_compound_kg": {"min": 0.005, "max": 0.02},
                    "coolant_consumption_l": {"min": 0.1, "max": 0.5},
                    "electricity_kwh": {"min": 0.05, "max": 0.2},
                    "time_min": {"min": 3, "max": 15},
                    "surface_roughness_um": {"min": 0.05, "max": 1.0},
                },
            },
            "condition": {
                "steel": "as needed for surface finish",
            },
            "note": {
                "aluminium": "Use non-ferrous tools with mandatory coolant",
            },
            "description": {
                "steel": "Achieve required surface finish and dimensional accuracy",
                "aluminium": "Final surface finishing using non-ferrous tools with coolant",
            },
        }
}

### Function: create_mechanical_process_exchanges


In [66]:
# ---- Mechanical-process flows -------------------------------------------------
# ecoinvent 3.12 has NO "grinding wheel" or "metal working fluid" dataset, and
# "abrasive blasting, alumina" is a service in m2 (not kg of abrasive). The
# flows below are therefore PROXIES, all in kg. Review/replace as needed.
def _pick(name, location, reference_product=None):
    """Exact-name ecoinvent lookup; fails loudly instead of returning a list."""
    hits = [
        a for a in ei
        if a["name"] == name and a["location"] == location
        and (reference_product is None or a.get("reference product") == reference_product)
    ]
    if len(hits) != 1:
        raise ValueError(f"{name!r} [{location}] gave {len(hits)} matches; refine the lookup")
    return hits[0]

mech_flows = {
    "electricity": electricity,                                   # kWh
    # abrasive grit (kg): alumina grit proxy (corundum), non-metallurgical grade
    "abrasive": _pick("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),
    # grinding wheel (kg): silicon carbide proxy (abrasive grain of wheels)
    "grinding_wheel": _pick("market for silicon carbide", "GLO"),
    # polishing compound (kg): alumina polishing powder proxy
    "polishing_compound": _pick("aluminium oxide production", "RoW", "aluminium oxide, non-metallurgical"),
    # coolant / metalworking fluid (kg): lubricating oil proxy (no emulsion dataset in ecoinvent)
    "coolant": _pick("market for lubricating oil", "RER"),
    # cutting tool material (kg): tungsten carbide powder proxy
    "tool_material": _pick("market for tungsten carbide powder", "GLO"),
}

# ---- Assumptions needed to convert the parameters to ecoinvent units ----------
COOLANT_DENSITY_KG_PER_L = 0.9   # assumed density of coolant
TOOL_MASS_KG = 0.01              # rule of thumb: ~10 g carbide indexable insert (typical 5-15 g),
                                 # consumed at 100 % wear; set to None to skip tool wear


def create_mechanical_process_exchanges(
    database_name,
    process_params,
    process_name,
    tool_mass_kg=None,
    coolant_density_kg_per_l=None,
):
    """Build Brightway exchanges from MECH_PREPOST_PARAMS.

    `process_params` is the material-specific dict (e.g. ["parameters"]["steel"]).
    Each parameter is a {"min", "max"} range (or a number); the exchange amount
    is the midpoint, with "minimum"/"maximum" kept for the scenario runs.
    Amounts are the totals consumed over one execution of the process (its
    duration, time_min), so the functional unit is 1 execution and no
    time scaling is applied.
    Parameters without an ecoinvent flow (time, area, roughness, ...) are ignored.
    """
    tool_mass_kg = TOOL_MASS_KG if tool_mass_kg is None else tool_mass_kg
    coolant_density_kg_per_l = COOLANT_DENSITY_KG_PER_L if coolant_density_kg_per_l is None else coolant_density_kg_per_l

    def as_key(activity_or_key):
        return getattr(activity_or_key, "key", activity_or_key)

    def bounds(value):
        if isinstance(value, dict):
            return float(value["min"]), float(value["max"])
        value = float(value)
        return value, value

    # parameter -> (flow, factor to the ecoinvent unit)
    parameter_map = {
        "electricity_kwh": ("electricity", 1.0),
        "abrasive_consumption_kg": ("abrasive", 1.0),
        "grinding_wheel_consumption_kg": ("grinding_wheel", 1.0),
        "polishing_compound_kg": ("polishing_compound", 1.0),
        "coolant_consumption_l": ("coolant", coolant_density_kg_per_l),
    }
    if tool_mass_kg is not None:
        parameter_map["tool_wear_pct"] = ("tool_material", tool_mass_kg / 100)  # % -> kg

    exchanges = [
        {
            "input": (database_name, process_name),
            "amount": 1,
            "type": "production",
        }
    ]

    ignored = []
    for parameter, value in process_params.items():
        if parameter not in parameter_map:
            ignored.append(parameter)
            continue
        flow_name, factor = parameter_map[parameter]
        low, high = (b * factor for b in bounds(value))
        exchanges.append({
            "input": as_key(mech_flows[flow_name]),
            "amount": (low + high) / 2,
            "type": "technosphere",
            "minimum": low,
            "maximum": high,
            "uncertainty_type": 1,
        })

    if tool_mass_kg is None and "tool_wear_pct" in ignored:
        print(f"WARNING {process_name}: tool_wear_pct skipped; set TOOL_MASS_KG to include tool wear")
    print(f"{process_name}: ignored (no ecoinvent flow): {ignored}")
    return exchanges


In [67]:
MechProcess_data = {}

for service_name, record in MECH_PREPOST_PARAMS.items():
    for material, material_params in record["parameters"].items():
        process_key = f"{service_name}_{material}"  # one process per service AND material
        MechProcess_data[(MECH_PREPOST_NAME, process_key)] = {
            "name": f"{service_name} for {material}",
            "unit": "unit",
            "location": "NL",
            "exchanges": create_mechanical_process_exchanges(
                MECH_PREPOST_NAME,
                material_params,
                process_key,
            ),
        }

MECH_PREPOST.write(MechProcess_data)  # uncomment once the proxies/assumptions above are confirmed


mechanical_treatment_steel: ignored (no ecoinvent flow): ['time_min', 'surface_area_m2']
mechanical_treatment_aluminium: ignored (no ecoinvent flow): ['time_min', 'surface_area_m2']
machining_steel: ignored (no ecoinvent flow): ['machining_time_min']
grinding_polishing_steel: ignored (no ecoinvent flow): ['time_min', 'surface_roughness_um']
grinding_polishing_aluminium: ignored (no ecoinvent flow): ['time_min', 'surface_roughness_um']


100%|██████████| 5/5 [00:00<00:00, 57456.22it/s]

10:48:05+0100 [info     ] Vacuuming database            


## LCIA results

In [68]:
def calculate_lcia_scenarios(process_databases, ef31_methods, cost_factors=None):
    if cost_factors is None:
        cost_factors = COST_FACTORS

    scenario_results = []

    if not isinstance(process_databases, (list, tuple)):
        process_databases = [process_databases]

    scenarios = ["minimum", "amount", "maximum"]
    scenario_keys = {
        "minimum": ["minimum", "min"],
        "amount": ["amount"],
        "maximum": ["maximum", "max"]
    }

    for database in process_databases:
        for process in database:
            print(process)
            process_exchanges = list(process.exchanges())

            for scenario in scenarios:
                original_amounts = [
                    (exchange, exchange["amount"])
                    for exchange in process_exchanges
                ]

                try:
                    for exchange, original_amount in original_amounts:
                        scenario_amount = original_amount
                        for key in scenario_keys[scenario]:
                            if key in exchange:
                                scenario_amount = exchange[key]
                                break

                        if scenario_amount != original_amount:
                            exchange["amount"] = scenario_amount
                            exchange.save()

                    # Calculate LCIA
                    lca = LCA({process: 1}, method=ef31_methods[0])
                    lca.lci()
                    lca.lcia()

                    # Calculate foreground electricity consumption and total cost
                    electricity_kwh = sum(
                        exchange["amount"]
                        for exchange in process_exchanges
                        if exchange["type"] == "technosphere"
                        and "electricity" in str(exchange.input).lower()
                    )
                    opex = calculate_process_cost(process, scenario, cost_factors)

                    for method in ef31_methods:
                        lca.switch_method(method)
                        lca.lcia()

                        method_tuple = str(method).replace("'", '').strip("()")
                        method_parts = [p.strip() for p in method_tuple.split(",")]
                        impact_category = method_parts[2] if len(method_parts) > 2 else str(method)

                        scenario_results.append({
                            "Database": database.filename,
                            "Process": process["name"],
                            "Key": process.key[1],
                            "Scenario": scenario,
                            "Method": str(method),
                            "Impact Category": impact_category,
                            "Amount": lca.score,
                            "Electricity Consumption (kWh)": electricity_kwh,
                            "Consumable costs (EUR)": opex
                        })
                finally:
                    for exchange, original_amount in original_amounts:
                        if exchange["amount"] != original_amount:
                            exchange["amount"] = original_amount
                            exchange.save()

    # Create DataFrame and add costs
    print(scenario_results)
    scenario_df = pd.DataFrame(scenario_results)

    pivot_scenario_df = scenario_df.pivot_table(
        index=["Database", "Process", "Key", "Scenario"],
        columns="Impact Category",
        values="Amount",
        aggfunc="sum"
    ).reset_index()

    result_index = ["Database", "Process", "Key", "Scenario"]
    resource_df = scenario_df[
        result_index + ["Electricity Consumption (kWh)", "Consumable costs (EUR)"]
    ].drop_duplicates(subset=result_index)

    final_df = pd.merge(pivot_scenario_df, resource_df,
                       on=result_index, how="left")

    print("\n\n=== EF 3.1 LCIA Results with Foreground Electricity Consumption and Costs ===")
    print(final_df.to_markdown(index=False, floatfmt=".4e"))

    # Save results
    if len(process_databases) == 1:
        db_name = process_databases[0].filename.split('.')[0]
    else:
        db_name = "_".join(db.filename.split('.')[0] for db in process_databases)

    output_file = f"results/{db_name}_LCIA_results.csv"
    json_output_file = f"results/{db_name}_LCIA_results.json"

    final_df.to_csv(output_file, index=False)
    final_df.to_json(json_output_file, orient="records", indent=2)

    print(f"\nScenario results saved to {output_file} and {json_output_file}")

    return final_df

### Call: calculate_lcia_scenarios


In [69]:
calculate_lcia_scenarios([CHEM_PREPOST, MECH_PREPOST], ef31_methods)

'alkaline_degreasing for steel' (unit, NL, None)


/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.65e+12)
  warnings.warn(msg, UmfpackWarning)


'passivation for steel' (unit, NL, None)
'alkaline_degreasing for aluminium' (unit, NL, None)
'pickling for steel' (unit, NL, None)
'mechanical_treatment for steel' (unit, NL, None)
'grinding_polishing for aluminium' (unit, NL, None)
'machining for steel' (unit, NL, None)
'grinding_polishing for steel' (unit, NL, None)
'mechanical_treatment for aluminium' (unit, NL, None)
[{'Database': 'Chemical_Pre_and_Post_processes.9a629b55', 'Process': 'alkaline_degreasing for steel', 'Key': 'alkaline_degreasing_steel', 'Scenario': 'minimum', 'Method': "('ecoinvent-3.12', 'EF v3.1', 'acidification', 'accumulated exceedance (AE)')", 'Impact Category': 'acidification', 'Amount': 0.0002281418707511578, 'Electricity Consumption (kWh)': 0.04839849046696124, 'Consumable costs (EUR)': 0.014035562235418759}, {'Database': 'Chemical_Pre_and_Post_processes.9a629b55', 'Process': 'alkaline_degreasing for steel', 'Key': 'alkaline_degreasing_steel', 'Scenario': 'minimum', 'Method': "('ecoinvent-3.12', 'EF v3.1', 

,Database,Process,Key,Scenario,acidification,climate change: total (EF v3.1 - IPCC 2021),eutrophication: terrestrial,human toxicity: carcinogenic,particulate matter formation,photochemical oxidant formation: human health,Electricity Consumption (kWh),Consumable costs (EUR)
0,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,amount,0.000884,0.198565,0.001618,2.127498e-09,5.891000e-09,0.000495,0.150035,0.043510
1,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,maximum,0.001532,0.344737,0.002818,3.795154e-09,1.033873e-08,0.000862,0.241992,0.070178
2,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for aluminium,alkaline_degreasing_aluminium,minimum,0.000236,0.052393,0.000418,4.598414e-10,1.443271e-09,0.000129,0.058078,0.016843
3,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,amount,0.000725,0.162395,0.001320,1.666894e-09,4.786442e-09,0.000404,0.133096,0.038598
4,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,maximum,0.001223,0.273983,0.002230,2.855052e-09,8.105044e-09,0.000683,0.217793,0.063160
5,Chemical_Pre_and_Post_processes.9a629b55,alkaline_degreasing for steel,alkaline_degreasing_steel,minimum,0.000228,0.050807,0.000410,4.787372e-10,1.467840e-09,0.000126,0.048398,0.014036
6,Chemical_Pre_and_Post_processes.9a629b55,passivation for steel,passivation_steel,amount,0.009266,1.050288,0.037359,3.361823e-10,4.630664e-08,0.002288,0.108897,0.031580
7,Chemical_Pre_and_Post_processes.9a629b55,passivation for steel,passivation_steel,maximum,0.013988,1.594648,0.056175,5.097850e-10,6.975380e-08,0.003475,0.217793,0.063160
8,Chemical_Pre_and_Post_processes.9a629b55,passivation for steel,passivation_steel,minimum,0.004543,0.505927,0.018543,1.625796e-10,2.285948e-08,0.001101,0.000000,0.000000
9,Chemical_Pre_and_Post_processes.9a629b55,pickling for steel,pickling_steel,amount,0.002222,0.408350,0.003282,2.552482e-09,1.510154e-08,0.001091,0.131919,0.038256


## Utilities

#### Proxy Checks

In [70]:
# Flow variables are defined by the ecoinvent searches run in other cells
# (using find_ecoinvent_activity). Declare any that are missing as None so
# the summary below never fails with a NameError and flags them as 🔴.
FLOW_VARIABLES = {
    "electricity": electricity,
    "water": water,
    "chemical_alkaline": chemical_alkaline,
    "acid_pickling": acid_pickling,
    "nitric_acid": nitric_acid,
    "abrasive": abrasive,
    "grinding_wheel": grinding_wheel,
    "polishing_compound": polishing_compound,
    "coolant": coolant,
    "waste_treatment": waste_treatment,
    "waste_paint_treatment": waste_paint_treatment,
    "waste_spent_solvent_treatment": waste_spent_solvent_treatment,
    "waste_spent_solvent_energy_recovery": waste_spent_solvent_energy_recovery,
}

def summarize_flows(flow_variables):
    rows = []
    for var_name, result in flow_variables.items():
        if result is None:
            status, n, selected, locations, units, note = (
                "🔴", 0, "-", "-", "-", "Not found: revise search terms"
            )
        elif isinstance(result, list):
            n = len(result)
            names = sorted({a.get("name") for a in result})
            units = ", ".join(sorted({str(a.get("unit")) for a in result}))
            locations = ", ".join(sorted({str(a.get("location")) for a in result})[:5])
            if len(result) > 5:
                locations += ", ..."
            selected = f"{len(names)} distinct name(s): {names[0][:60]}" + ("..." if len(names) > 1 else "")
            status = "🟡"
            note = "Ambiguous: refine terms or choose a location"
            if len(names) > 1 or len(units.split(", ")) > 1:
                status = "🔴"
                note = "Mixed products/units: likely wrong matches"
        else:
            n, status = 1, "🟢"
            selected = result.get("name")
            locations = result.get("location")
            units = result.get("unit")
            note = "OK"
        rows.append({
            "Status": status,
            "Variable": var_name,
            "Matches": n,
            "Selected / Name": selected,
            "Location(s)": locations,
            "Unit(s)": units,
            "Comment": note,
        })
    return pd.DataFrame(rows)

def style_status(df):
    colors = {"🟢": "#d4edda", "🟡": "#fff3cd", "🔴": "#f8d7da"}
    return (
        df.style
        .apply(lambda row: [f"background-color: {colors[row['Status']]}"] * len(row), axis=1)
        .hide(axis="index")
        .set_properties(**{"text-align": "left"})
    )

flow_summary = summarize_flows(FLOW_VARIABLES)
display(style_status(flow_summary))


counts = flow_summary["Status"].value_counts()
print(
    f"🟢 OK: {counts.get('🟢', 0)} | "
    f"🟡 Ambiguous: {counts.get('🟡', 0)} | "
    f"🔴 Needs attention: {counts.get('🔴', 0)}"
)

Status,Variable,Matches,Selected / Name,Location(s),Unit(s),Comment
🟢,electricity,1,"market group for electricity, medium voltage",RER,kilowatt hour,OK
🟢,water,1,market group for tap water,RER,kilogram,OK
🟢,chemical_alkaline,1,"market for sodium hydroxide, without water, in 50% solution state",RER,kilogram,OK
🟢,acid_pickling,1,"market for hydrochloric acid, without water, in 30% solution state",RER,kilogram,OK
🟢,nitric_acid,1,"market for nitric acid, without water, in 50% solution state",RER w/o RU,kilogram,OK
🟢,abrasive,1,"market for abrasive blasting, alumina, carbon steel substrate",GLO,square meter,OK
🔴,grinding_wheel,0,-,-,-,Not found: revise search terms
🔴,polishing_compound,25,3 distinct name(s): aluminium oxide production...,"CA, CN, IAI Area, Asia, without China and GCC, IAI Area, Gulf Cooperation Council, IAI Area, Russia and Eastern Europe, ...","kilogram, square meter",Mixed products/units: likely wrong matches
🔴,coolant,0,-,-,-,Not found: revise search terms
🟡,waste_treatment,3,"1 distinct name(s): treatment of waste steel, inert material landfill","CH, Europe without Switzerland, RoW",kilogram,Ambiguous: refine terms or choose a location


🟢 OK: 6 | 🟡 Ambiguous: 1 | 🔴 Needs attention: 6
